# 자소서 질문 군집화 분석
**목적**: Sentence-BERT + K-Means로 질문 유형을 자동 분류하고, 군집별 평가 모델을 결정한다.

| 단계 | 내용 |
|------|------|
| Step 1 | 데이터 로드 및 전처리 |
| Step 2 | SBERT 임베딩 |
| Step 3 | TF-IDF vs SBERT 비교 |
| Step 4 | Elbow Method |
| Step 5 | K-Means 군집화 + 대표 질문 |
| Step 6 | cluster_map.json 초안 저장 |
| Step 7 | 새 질문 예측 |
| Step 8 | 혼합 질문 처리 |
| Step 9 | relevance_detector 통합 테스트 |

In [ ]:
import sys, os
# 프로젝트 루트를 sys.path에 추가
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print("Project root:", project_root)

In [ ]:
from models.role05_match.question_clusterer import (
    load_questions,
    get_sbert_embeddings,
    compare_tfidf_sbert,
    plot_elbow,
    fit_kmeans,
    print_cluster_representatives,
    save_cluster_map,
    load_cluster_map,
    predict_cluster,
    predict_cluster_multi,
    get_evaluation_models,
)
print("모듈 임포트 완료")

---
## Step 1: 데이터 로드 및 전처리

In [ ]:
questions = load_questions()
print(f"유니크 질문 수: {len(questions):,}개")
print("\n샘플 5개:")
for q in questions[:5]:
    print(f"  - {q[:80]}")

---
## Step 2: Sentence-BERT 임베딩
> `jhgan/ko-sroberta-multitask` 모델 사용. 최초 실행 시 다운로드 (~400MB).

In [ ]:
embeddings = get_sbert_embeddings(questions)
print(f"임베딩 shape: {embeddings.shape}  (질문 수 × 768차원)")

---
## Step 3: TF-IDF vs Sentence-BERT 비교
> "팀 갈등 해결 경험"과 "협업 중 어려움을 극복한 사례"가 같은 군집에 묶이는지 확인

In [ ]:
km_tfidf, km_sbert_cmp, vectorizer = compare_tfidf_sbert(questions, embeddings, k=7)

---
## Step 4: Elbow Method — 최적 k 결정

In [ ]:
optimal_k = plot_elbow(embeddings, k_range=range(2, 11))
print(f"\n최적 k = {optimal_k}")

> **직접 설정할 수도 있습니다.**  
> Elbow 추천이 마음에 안 들면 아래 셀에서 `K`를 변경하세요.

In [ ]:
K = optimal_k  # 여기를 수동으로 바꿔도 됩니다 (예: K = 6)

---
## Step 5: K-Means 군집화 + 대표 질문 출력
> 아래 출력을 보고 각 군집에 이름을 붙여주세요 → Step 6에서 cluster_map.json에 반영됩니다.

In [ ]:
km = fit_kmeans(embeddings, k=K)
cluster_info = print_cluster_representatives(km, questions, embeddings, top_n=5)

---
## Step 6: cluster_map.json 초안 저장
> 저장 후 `models/role05_match/cluster_map.json`을 열어 각 군집의  
> `name` / `models` / `star_required` 를 직접 수정하세요.

In [ ]:
save_cluster_map(cluster_info, k=K)

print("\n수정 예시 (cluster_map.json에서 직접 편집):")
example = {
  "cluster_0": {
    "name": "경험형",
    "models": ["star", "hedge", "self"],
    "star_required": True
  },
  "cluster_1": {
    "name": "포부형",
    "models": ["clarity", "hedge"],
    "star_required": False
  },
  "cluster_2": {
    "name": "지원동기형",
    "models": ["clarity", "hedge"],
    "star_required": False
  }
}
import json; print(json.dumps(example, ensure_ascii=False, indent=2))

---
## Step 7: 새 질문 입력 시 군집 예측

In [ ]:
cmap = load_cluster_map()

test_questions = [
    "팀 갈등 해결 경험에 대해 서술하시오",
    "협업 중 어려움을 극복한 사례를 작성하시오",
    "지원 동기를 기술하시오",
    "입사 후 포부를 서술하시오",
    "본인의 강점과 약점을 작성하시오",
]

print(f"{'질문':45s} {'군집':6s} {'이름':15s} {'모델'} ")
print("-" * 85)
for q in test_questions:
    res   = predict_cluster(q, km=km, cluster_map=cmap)
    cinfo = res['cluster_info']
    name  = cinfo.get('name', '-')
    models = ",".join(cinfo.get('models', []))
    print(f"{q[:44]:45s} {res['cluster_id']:6d} {name:15s} {models}")

---
## Step 8: 혼합 질문 처리

In [ ]:
mixed_questions = [
    "경험을 바탕으로 입사 후 포부를 서술하시오",
    "성장 과정과 지원 동기를 함께 작성하시오",
    "① 팀 프로젝트 경험  ② 향후 목표",
]

for mq in mixed_questions:
    res = predict_cluster_multi(mq, km=km, cluster_map=cmap)
    print(f"원본 : {res['original_question']}")
    print(f"분리 : {res['subquestions']}")
    print(f"군집 : {res['cluster_ids']}")
    print(f"모델 : {res['merged_models']}")
    print(f"STAR : {res['star_required']}")
    print("-" * 60)

---
## Step 9: relevance_detector.py 통합 테스트
> 기존 `detect_question_intents()` API가 군집화 결과를 반환하는지 확인

In [ ]:
from models.role05_match.relevance_detector import (
    detect_question_intents,
    detect_question_intents_multi,
)

samples = [
    "팀 갈등 해결 경험에 대해 서술하시오",
    "협업 중 어려움을 극복한 사례를 작성하시오",
    "지원 동기를 기술하시오",
    "경험을 바탕으로 입사 후 포부를 서술하시오",  # 혼합 질문
]

print("detect_question_intents() 결과:")
print("=" * 65)
for q in samples:
    res = detect_question_intents(q)
    print(f"Q: {q}")
    print(f"   intent={res['intent']}  method={res['method']}")
    print(f"   models={res['models']}  star={res['star_required']}")
    print()

In [ ]:
# 규칙 기반 폴백 테스트 (use_clustering=False)
print("규칙 기반 폴백 테스트:")
print("=" * 65)
for q in samples[:3]:
    res = detect_question_intents(q, use_clustering=False)
    print(f"Q: {q}")
    print(f"   intent={res['intent']}  method={res['method']}  models={res['models']}")
    print()

---
## 결과 요약

| 항목 | 값 |
|------|----|
| 유니크 질문 수 | (위 셀 확인) |
| 최적 k | (위 셀 확인) |
| 사용 모델 | `jhgan/ko-sroberta-multitask` |
| 캐시 위치 | `models/role05_match/cache/` |
| 군집 매핑 파일 | `models/role05_match/cluster_map.json` |

### 다음 할 일
1. `cluster_map.json` 열어서 각 군집에 이름 붙이기 (`name` 필드)
2. `models`, `star_required` 조정
3. (선택) `relevance_detector.py`의 `_CLUSTER_TO_INTENT` 딕셔너리에  
   `cluster_id → 기존 intent 레이블` 매핑 추가